
# Dataflow Flex Templates — Hands-On Lab
### GCP Training Program — Apache Beam Model & Dataflow Templates

## Problem Statement

A data team has a small but genuinely useful Beam pipeline — it reads a text file and counts word
frequency. Right now, running it means checking out the source code, having a matching Python
environment, and running `python pipeline.py` by hand. A teammate on another team, or a scheduled
job, can't easily reuse it without doing all of that themselves.

**This notebook packages that pipeline once as a Dataflow Flex Template** — a container image plus
a small JSON spec — so afterward, anyone with the right IAM permissions can launch it with just a
job name and two parameters (an input path and an output path), no source code or Python
environment required on their end.

## What You'll Do, and How Long It Takes

| Step | What happens | Approx. time |
|---|---|---|
| 1-2 | Auth, configure, enable APIs, create bucket + Artifact Registry repo | 1-2 min |
| 3 | Write the pipeline, requirements, and metadata files | under 1 min |
| 4 | **Build the Flex Template** (Cloud Build containerizes the pipeline) | 3-6 min |
| 5 | **Run the Flex Template** as a real Dataflow job | 5-10 min |
| 6 | Verify the output | 1 min |
| 7 | Cleanup | 2-3 min |

**Total: roughly 15-20 minutes**, with Steps 4 and 5 as the two variable-length waits — same
pattern as every other "provisioning-heavy" step we've hit elsewhere in this course (Cloud Build
containerization, then Dataflow worker startup). Nothing here is faster to skip; both waits are the
actual mechanism being demonstrated, not overhead to avoid.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs against
your own project.


## 1. Authenticate This Session

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")



## 2. Configure Your Project & Create Supporting Resources

Creates a GCS bucket (for the input file, template spec, and output) and an Artifact Registry
repository (for the container image) — both one-time, reusable resources.


In [ ]:

import time

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
REGION = input("Enter your region [default: us-central1]: ").strip() or "us-central1"

_suffix = int(time.time())
BUCKET_NAME = f"{PROJECT_ID}-flex-template-lab-{_suffix}"
BUCKET_URI = f"gs://{BUCKET_NAME}"
REPO_NAME = "dataflow-templates"
IMAGE_NAME = f"wordcount-{_suffix}"

!gcloud config set project {PROJECT_ID}
!gcloud services enable dataflow.googleapis.com cloudbuild.googleapis.com \
    artifactregistry.googleapis.com compute.googleapis.com --project={PROJECT_ID}

!gsutil mb -l {REGION} {BUCKET_URI}

!gcloud artifacts repositories create {REPO_NAME} \
    --repository-format=docker --location={REGION} --project={PROJECT_ID} \
    --description="Dataflow Flex Template images for training lab" --quiet

print("Project:", PROJECT_ID)
print("Region:", REGION)
print("Bucket:", BUCKET_URI)
print("Artifact Registry repo:", REPO_NAME)



> 🖥️ **Check it in the console:** **Cloud Storage > Buckets** → your new bucket is listed.
> **Artifact Registry > Repositories** → `dataflow-templates` is listed, empty for now — Section 4
> will push an image here.



## 3. Write the Pipeline, Requirements, and Metadata

### 3.1 A Small Input File to Process


In [ ]:

sample_text = '''Apache Beam is a unified model for batch and streaming data processing.
Dataflow runs Apache Beam pipelines on managed infrastructure.
A Flex Template packages a Beam pipeline as a container image.
Anyone can launch a Flex Template without needing the pipeline source code.
Apache Beam Apache Beam Dataflow Dataflow Dataflow
'''

with open("input.txt", "w") as f:
    f.write(sample_text)

!gsutil cp input.txt {BUCKET_URI}/input.txt
print(f"Uploaded input file to {BUCKET_URI}/input.txt")



### 3.2 The Pipeline Itself
**Note the two things that make this Flex-Template-ready, not just an ordinary Beam script:**
`add_value_provider_argument` (so `input_path`/`output_path` are settable at *launch* time, not
baked in now), and calling `pipeline.run()` directly rather than the `with beam.Pipeline(...) as p:`
context manager (which would block waiting for job completion — fatal for a template launcher,
which must exit immediately after submitting the job).


In [ ]:

import os
os.makedirs("flex_template", exist_ok=True)

with open("flex_template/pipeline.py", "w") as f:
    f.write('''
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions


class TemplateOptions(PipelineOptions):
    @classmethod
    def _add_argparse_args(cls, parser):
        parser.add_value_provider_argument("--input_path", type=str, help="GCS path to input text file")
        parser.add_value_provider_argument("--output_path", type=str, help="GCS path prefix for output")


def run():
    pipeline_options = PipelineOptions()
    template_options = pipeline_options.view_as(TemplateOptions)

    pipeline = beam.Pipeline(options=pipeline_options)
    (
        pipeline
        | "ReadLines" >> beam.io.ReadFromText(template_options.input_path)
        | "ExtractWords" >> beam.FlatMap(lambda line: line.split())
        | "CountWords" >> beam.combiners.Count.PerElement()
        | "FormatResult" >> beam.Map(lambda kv: f"{kv[0]}: {kv[1]}")
        | "WriteResult" >> beam.io.WriteToText(template_options.output_path)
    )
    # IMPORTANT: call run() and let the script exit -- do NOT use
    # `with beam.Pipeline(...) as p:` or call wait_until_finish(), since that
    # blocks and prevents the Flex Template launcher from completing.
    pipeline.run()


if __name__ == "__main__":
    run()
''')

print("Wrote flex_template/pipeline.py")


In [ ]:

with open("flex_template/requirements.txt", "w") as f:
    f.write("apache-beam[gcp]==2.60.0\n")

print("Wrote flex_template/requirements.txt")


### 3.3 The Template Metadata

In [ ]:

import json

metadata = {
    "name": "Word Count Flex Template",
    "description": "Reads a text file, counts word frequency, writes the result to GCS.",
    "parameters": [
        {
            "name": "input_path",
            "label": "Input file",
            "helpText": "GCS path to the input text file",
            "isOptional": False,
        },
        {
            "name": "output_path",
            "label": "Output prefix",
            "helpText": "GCS path prefix for the output files",
            "isOptional": False,
        },
    ],
}

with open("flex_template/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Wrote flex_template/metadata.json")
print(json.dumps(metadata, indent=2))



## 4. Build the Flex Template

**What this does:** containerizes `pipeline.py` + `requirements.txt` (no hand-written Dockerfile
needed for a plain Python pipeline — `--py-path` plus `--flex-template-base-image PYTHON3` handles
that automatically), pushes the image to Artifact Registry, and writes the template spec JSON to
GCS. **This is the slower of the two waits in this notebook — expect 3-6 minutes.**


In [ ]:

TEMPLATE_SPEC_PATH = f"{BUCKET_URI}/templates/wordcount.json"
IMAGE_PATH = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME}:latest"

!gcloud dataflow flex-template build {TEMPLATE_SPEC_PATH} \
    --image-gcr-path {IMAGE_PATH} \
    --sdk-language PYTHON \
    --flex-template-base-image PYTHON3 \
    --metadata-file flex_template/metadata.json \
    --py-path flex_template \
    --env FLEX_TEMPLATE_PYTHON_PY_FILE=pipeline.py \
    --env FLEX_TEMPLATE_PYTHON_REQUIREMENTS_FILE=requirements.txt \
    --project={PROJECT_ID}

print("\nTemplate spec written to:", TEMPLATE_SPEC_PATH)
print("Container image:", IMAGE_PATH)



> 🖥️ **Check it in the console:** **Cloud Build > History** → a build should appear and complete
> (this is what just containerized your pipeline). **Artifact Registry > dataflow-templates repo**
> → the pushed image is listed. **Cloud Storage > your bucket > templates/** → `wordcount.json`
> is the template spec this build just wrote.



## 5. Run the Flex Template

**What this does:** launches an actual Dataflow job from the template, passing `input_path` and
`output_path` as runtime parameters — exactly what a teammate or scheduler would do, with no
access to `pipeline.py` at all. **This is the second slower wait — expect 5-10 minutes** for a
small batch job to spin up a worker, run, and shut down.


In [ ]:

JOB_NAME = f"wordcount-run-{_suffix}"
OUTPUT_PREFIX = f"{BUCKET_URI}/output/result"

!gcloud dataflow flex-template run {JOB_NAME} \
    --template-file-gcs-location {TEMPLATE_SPEC_PATH} \
    --region {REGION} \
    --parameters input_path={BUCKET_URI}/input.txt \
    --parameters output_path={OUTPUT_PREFIX} \
    --project={PROJECT_ID}

print(f"\nLaunched job: {JOB_NAME}")
print("Poll status with the next cell, or watch it in the console (link below).")


In [ ]:

import time

print("Polling job status every 30 seconds (Dataflow jobs are not instantaneous)...")
while True:
    state = !gcloud dataflow jobs list --region={REGION} --project={PROJECT_ID} \
        --filter="name={JOB_NAME}" --format="value(state)"
    state = state[0] if state else "UNKNOWN"
    print(f"  Job state: {state}")
    if state in ("JOB_STATE_DONE", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED"):
        break
    time.sleep(30)

print("\nFinal state:", state)



> 🖥️ **Check it in the console:** **Dataflow > Jobs > wordcount-run-...** → the **Job graph** tab
> shows each labeled stage from `pipeline.py` (`ReadLines` → `ExtractWords` → `CountWords` →
> `FormatResult` → `WriteResult`) as a visual DAG — click any stage to see its throughput once the
> job is running, and its logs if anything fails.



## 6. Verify the Output

**What to look for:** word counts matching the input text — `apache: 3`, `beam: 3`,
`dataflow: 4`, etc., proving the templated pipeline actually processed the file end-to-end.


In [ ]:

!gsutil cat {OUTPUT_PREFIX}-* 2>/dev/null || echo "Output not ready yet — wait for JOB_STATE_DONE above, then re-run this cell."



> 🖥️ **Check it in the console:** **Cloud Storage > your bucket > output/** → one or more
> `result-*-of-*` files should be listed — click one to preview its contents directly in the
> console, without needing `gsutil cat`.



## 7. Cleanup

Removes the container image, the Artifact Registry repository, and the bucket (which holds the
template spec, input, and output). The Dataflow job itself doesn't need deleting — a completed
batch job simply stops consuming resources; if it's still running for any reason, this cancels it
first.


In [ ]:

CONFIRM_DELETE = False  # <-- set to True to actually delete resources

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing will be deleted. "
          "Set CONFIRM_DELETE = True above and re-run this cell when you're ready.")


In [ ]:

if CONFIRM_DELETE:
    try:
        !gcloud dataflow jobs cancel {JOB_NAME} --region={REGION} --project={PROJECT_ID} --quiet
        print("Cancelled job (no-op if it already completed).")
    except Exception as e:
        print("Job cancel skipped/failed (likely already done):", e)

    try:
        !gcloud artifacts repositories delete {REPO_NAME} --location={REGION} \
            --project={PROJECT_ID} --quiet
        print(f"Deleted Artifact Registry repo: {REPO_NAME}")
    except Exception as e:
        print("Artifact Registry cleanup skipped/failed:", e)

    try:
        !gsutil -m rm -r {BUCKET_URI}
        print(f"Deleted bucket: {BUCKET_URI}")
    except Exception as e:
        print("Bucket cleanup skipped/failed:", e)

    print("\nCleanup complete.")



> 🖥️ **Final console check:** **Cloud Storage > Buckets**, **Artifact Registry > Repositories**,
> and **Dataflow > Jobs** should show no resources remaining from this notebook (the Dataflow job
> will still be listed as history — jobs are never deleted, only shown as completed/cancelled —
> but nothing from it should still be running or billing).
